In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

#ANON_ANON_[REDACTED]
%matplotlib inline
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)




In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()


In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df=df, target_column="Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
# droped and idont know why!

In [ ]:
# Task 2: Write your code here:
# Check how many missing values exist in each column
df.isnull().sum()


# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns
# we will Fill missing numeric values with the mean
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())
    #we will Fill missing with categorical values with the most frequent value (mode)
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

    # Verify that there are no missing values left
df.isnull().sum()



In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
df.describe()

In [ ]:
# Task 4: Write your code here:
# Separate  y
y = df['Delivery_Time']

# Separate x

X = df.drop(columns=['Delivery_Time'])

X = pd.get_dummies(X, drop_first=True, dtype=int)
X.head()


In [ ]:
# Task 5: Write your code here:


scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled[:5]

In [ ]:
# Task 6: Write your code here:
# Target imbalance check is not required
# Delivery_Time is a continuous variable (regression problem),
# so class imbalance does not apply here:).

In [ ]:
# Task 1: Write your code here:
# detect target variable and split
y = df['Delivery_Time']

X = df.drop(columns=['Delivery_Time'])

In [ ]:
# Task 2,3,4,5: Write your code here:

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    X_train = pd.get_dummies(X_train, drop_first=True, dtype=int)
    X_val = pd.get_dummies(X_val, drop_first=True, dtype=int)
    X_train, X_val = X_train.align(X_val, join='left', axis=1, fill_value=0)

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    mae_scores.append(mean_absolute_error(y_val, y_pred))

print("Average MAE across folds:", np.mean(mae_scores))

In [ ]:
# Task 1: Write your code here:

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

model = RandomForestRegressor(random_state=42)
model.fit(X_encoded, y)



importances = model.feature_importances_
feature_names = X_encoded.columns

indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), feature_names[indices], rotation=90)
plt.title("Feature Importance from RandomForest")
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:


y_pred = model.predict(X_encoded)

plt.figure(figsize=(8, 5))
plt.hist(y_pred, bins=30)
plt.xlabel("Predicted Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Histogram of Predicted Delivery Time")
plt.show()

In [ ]:
# Task Bonus: Write your code here:
